# Ortak canlı demo

Görüntü seçip RGB/HSV maskelerini, alan/baş örtüşmesini ve ortak sınıf kararını göstermek için entegrasyon akışı. Henüz gerçek veri ve üye uygulamaları yoktur. Eksik eşikler, dondurulmamış kontrol ayarları ve uygulanmamış modüller açıkça bildirilir. Önizleme nihai sonuç tablosuna otomatik yazılmaz.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook'u klonlanan depo içinden açın.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common import load_config, load_dataset
records = load_dataset()
print(f"Kaynak listesinde {len(records)} görüntü var.")
if not records:
    print("Veri henüz eklenmedi. Bu dosya deney başlangıç şablonudur.")


## Gösterim

Veri ve üye kodları tamamlandığında görüntüyü seçip Çalıştır'a basın. Öğretim elemanının seçtiği görüntü aynı akışla gösterilir.


In [ ]:
def show_demo(image_id):
    import matplotlib.pyplot as plt
    import pandas as pd
    from IPython.display import display
    from src.demo import run_color_demo

    result = run_color_demo(image_id)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(result["image"][:, :, ::-1])
    axes[0].set_title(image_id)
    for axis, method in zip(axes[1:], ("rgb", "hsv")):
        if method in result["masks"]:
            axis.imshow(result["masks"][method], cmap="gray", vmin=0, vmax=255)
        else:
            axis.text(0.5, 0.5, "Yöntem/ayar henüz hazır değil", ha="center")
        axis.set_title(method.upper())
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    for message in result["messages"]:
        print(message)
    if result["rows"]:
        display(pd.DataFrame(result["rows"]))
    else:
        print("Tamamlanmış yöntem olmadığı için ölçüm tablosu üretilmedi.")


In [ ]:
if records:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    selector = widgets.Dropdown(options=[row["image_id"] for row in records],
                                description="Görüntü:")
    button = widgets.Button(description="Çalıştır")
    output = widgets.Output()

    def on_run(_):
        with output:
            clear_output(wait=True)
            try:
                show_demo(selector.value)
            except (ValueError, OSError, KeyError, ImportError) as error:
                print(f"Demo henüz hazır değil: {error}")

    button.on_click(on_run)
    display(widgets.HBox([selector, button]), output)
else:
    print("Seçilebilir görüntü yok. Önce data/sources.csv, gerçek görüntüler ve etiketleri ekleyin.")


## Sunumdan önce

Altı veri kaydını doğrulayın; RGB/HSV ve ortak karar kuralını geliştirme setinde dondurun. Üye uygulamalarını tamamlayıp ortak demo üzerinden gerçekten çalıştırın. Slaytlar ayrı hazırlanır; bu notebook slayt yerine kullanılmaz.
